# Organizando experimentos

## 1 Convenções

Quando trabalhamos em equipe e/ou a medida que os projetos crescem, se torna necessária a adoção de convenções para organização do histórico de projetos no MLFlow. Dessa forma, o histórico será de fato um registro documental que permite a rastreabilidade do que foi avaliado e como o desenvolvimento aconteceu. Os principais itens que compõem uma boa _checklist_ de rastreabilidade são:

1. **Objetivos da execução:** Deve ficar claro o que se pretendia descobrir ou validar, qual era a hipótese testada.
2. **Origem dos dados:** Registrar qual base de dados foi utilizada (versão do dataset).
3. **Versão do código:** Associar a execução a um commit do Git, permitindo saber exatamente qual código gerou aquele resultado.
4. **Pré-processamento:** Quais filtros ou técnicas de transformação foram aplicados aos dados.
5. **Estratégia de avaliação:** Documentar como foi feita a divisão dos dados (estratégia de split).
6. **Resultados:** As métricas obtidas e os artefatos gerados (gráficos, relatórios) que auxiliam na interpretação.
7. **Decisão:** Registrar qual foi a decisão tomada após a análise daquela execução (por exemplo, se o modelo foi descartado, se é um candidato ou se tornou o campeão).

### 1.1 Experimentos

O experimentos agrupam _runs_ por projeto, fase e objetivo. A recomendação é utilizar uma estrutura que responda rapidamente a três perguntas: _**"qual o projeto"**_, _**"qual a fase"**_ e _**"qual o objetivo"**_ daquela bateria de testes.

O padrão pode ser algo como:

> `<projeto>_<fase>_<objetivo>`

O importante é que, ao olhar para a lista, qualquer pessoa da equipe saiba identificar o contexto daquela série de experimentos.

### 1.2 _Runs_

Para as execuções, a convenção deve facilitar a leitura e a identificação do histórico. Uma prática recomendada é descrver: _**"a etapa do projeto"**_, _**"o modelo utilizado"**_, _**"o objetivo específico daquela rodada"**_ e _**"a versão daquela execução"**_. Algo como: aula, modelo, objetivo e versão. Isso cria uma trilha cronológica e lógica do desenvolvimento.

> `<etapa>_<modelo>_<objetivo>_<versao>`

### 1.3 _Tags_

Descrevem o contexto e permitem filtragens. A convenção aqui é separar as _tags_ por grupos de responsabilidade:

- **Projeto:** identifica a qual iniciativa a run pertence. Ex: `project`, `task`, `target`;
- **Dados:** armazena a versão do dataset utilizado. Ex: `dataset_version`, `feature_set`, `split_strategy`;
- **Código:** registra a versão do commit para garantir a reprodutibilidade. Ex: `code_version`, `git_commit`;
- **Experimento:** descreve a hipótese que está sendo testada. Ex: `hypothesis`, `model_name`, `model_family`, `tracking_type`, `run_porpuse`;
  - Como exemplo ruim de hipótese podemos citar: _“testar um random forest”_, que é um texto vago e não informa que critério foi usado.
  - Um texto mais adequado para o que buscamos é: _"avaliar se um modelo Random Forest melhora a métrica ROC-AUC e recall (sensibilidade) da classe inadimplente em relação ao baseline de Regressão Logística, aceitando menor interpretabilidade se o ganho técnico for relevante"_. Isso ajuda muito quando precisamos justificar por que executamos a _run_.
- **Decisão:** indica o status daquela execução, como candidato, desafiante ou campeão. Ex: `selection_status`, `selected_for_registry`, `selection_rule`.

Uma dica valiosa é centralizar essas _tags_ em um dicionário comum, que pode ser reutilizado, evitando repetição de código e erros de digitação que possam criar categorias duplicadas ou confusas.

In [1]:
# capturando commit hash do git para rastreabilidade do codigo
import subprocess

def get_git_commit(default="not_available"):
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            text=True,
        ).strip()
        return commit
    except Exception:
        return default


CODE_VERSION = get_git_commit()
CODE_VERSION

'230a791'

In [2]:
# define tags comuns para todas as runs, para melhor rastreabilidade do experimento
COMMON_TAGS = {
    "project":         "risco_credito",
    "task":            "binary_classification",
    "target":          "inadimplente",
    "dataset_version": "synthetic_credit_v1",
    "feature_set":     "core_features_v1",
    "split_strategy":  "train_test_stratified_75_25",
    "code_version":    CODE_VERSION,
}
COMMON_TAGS

{'project': 'risco_credito',
 'task': 'binary_classification',
 'target': 'inadimplente',
 'dataset_version': 'synthetic_credit_v1',
 'feature_set': 'core_features_v1',
 'split_strategy': 'train_test_stratified_75_25',
 'code_version': '230a791'}

### 1.4 Artefatos

Permitem o armazenamento de evidências para auditoria e colaboração. A convenção é organizar os arquivos de saída em pastas temáticas dentro da estrutura da execução. As pastas sugeridas são: 

```text
artifacts/
├── data/                                   # resumo das bases e versões dos dados processados
│   └── dataset_profile.json   
├── reports/                                # relatórios de desempenho em formato de texto ou markdown
│   ├── classification_report.txt
│   ├── run_summary.md
│   └── champion_summary.md
├── plots/                                  # gráficos e visualizações de avaliação
│   ├── matriz_confusao.png
│   ├── curva_roc.png
│   └── coeficientes_modelo.png
├── tables/                                 # tabelas reutilizáveis
│   └── coeficientes_modelo.json
└── selection                               # documentos ou registros que sustentam a decisão sobre a escolha do modelo campeão
    ├── decision_matrix.json
    └── registry_preparations.json
```

## 2 Elegendo o modelo campeão

O próximo passo é eleger um modelo campeão, o objetivo é selecionar o modelo mais adequado ao contexto atual, baseando essa escolha em critérios bem definidos.

Para isso devemos comparar: desempenho, interpretabilidade e custo operacional. Não devemos apenas escolher o modelo que possui o maior valor em uma métrica específica, mas sim decidir de acordo com critérios objetivos que equilibrem desempenho e operação, formando assim uma decisão embasada.

Mas primeiro, vamos levantar o servidor local do MLflow para que possamos acessar a _user interface_ . Para isso rodamos o comando abaixo no terminal

```terminal
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

e depois acessamos o endereço [http://127.0.0.1:5000](http://127.0.0.1:5000).

### 2.1 Selecionando as _runs_ de comparação

Dentre os modelos candidatos temos:

- **Regressão logística:** tende a ser mais simples, rápida e interpretável
- **Árvore de decisão:** também é relativamente simples, porém pode ser instável
- **Random Forest:** tende a ser mais robusto, mas é menos direto de explicar e pode ter custo operacional elevado.

Vamos começar estabelecendo o _tracking_ com o banco de dados e selecionar as _runs_ de comparação:

In [19]:
import mlflow

# define a ligacao com o banco de dados relacional ja existente, onde o MLflow vai armazenar os metadados do experimento
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# recupera experimento pelo nome
EXPERIMENT_NAME = "risco_credito_comparacao_modelos"
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# filtra runs do experimento de comparacao usando tags
df_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_purpose = 'compare_model_candidates'",
    order_by=["metrics.roc_auc_teste DESC"],
)

if df_runs.empty:
    raise ValueError("Nenhuma run para comparação foi encontrada para seleção.")

df_runs.shape

(3, 120)

Como o DataFrame das _runs_ de comparação possui mais de 100 colunas, precisamos reduzi-lo para conter apenas as colunas relevantes para a comparação, mantendo as métricas principais (ROC, _precision_, _recall_ e F1), o nome do modelo e identificação da _run_. E além disso, os nomes das colunas podem ser normalizados:

In [20]:
model_col     = ("params.model_name" if "params.model_name" in df_runs.columns else "tags.model_name") # determina coluna com nome do modelo
decision_cols = [
    "run_id",
    "tags.mlflow.runName",
    model_col,
    "metrics.roc_auc_teste",
    "metrics.roc_auc_cv_media",
    "metrics.roc_auc_cv_desvio",
    "metrics.accuracy_teste",
    "metrics.precision_inadimplente",
    "metrics.recall_inadimplente",
    "metrics.f1_inadimplente",
]

available_cols = [col for col in decision_cols if col in df_runs.columns] # garante que apenas colunas existentes no dataframe sejam selecionadas
df_decision    = df_runs[available_cols].copy()
df_decision    = df_decision.rename(
    columns={
        "tags.mlflow.runName": "run_name",
        model_col: "model_name",
        "metrics.roc_auc_teste": "roc_auc_teste",
        "metrics.roc_auc_cv_media": "roc_auc_cv_media",
        "metrics.roc_auc_cv_desvio": "roc_auc_cv_desvio",
        "metrics.accuracy_teste": "accuracy_teste",
        "metrics.precision_inadimplente": "precision_inadimplente",
        "metrics.recall_inadimplente": "recall_inadimplente",
        "metrics.f1_inadimplente": "f1_inadimplente",
    }
)

metric_cols = [
    "roc_auc_teste",
    "roc_auc_cv_media",
    "roc_auc_cv_desvio",
    "accuracy_teste",
    "precision_inadimplente",
    "recall_inadimplente",
    "f1_inadimplente",
]

df_decision[metric_cols] = df_decision[metric_cols].astype(float)
df_decision              = df_decision.sort_values("roc_auc_teste", ascending=False)

print(df_decision.shape)
df_decision

(3, 10)


,run_id,run_name,model_name,roc_auc_teste,roc_auc_cv_media,roc_auc_cv_desvio,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638


### 2.2 Política de seleção

Como a decisão será tomada de acordo com múltiplos critérios, faz sentido definir uma **política de seleção** em que são atribuídos pesos para cada um dos critérios (técnicos e operacionais) que reflitam sua importância relativa ao contexto do projeto. Dessa forma, atribuindo valores numéricos para cada critério, poderemos calcular um _score_ composto combinando fatores técnicos e operacionais.

Esse método permite que a decisão não se baseie apenas no desempenho absoluto de alguma métrica, mas sim no equilíbrio entre desempenho, interpretabilidade e simplicidade operacional. Lembrando que a definição dos pesos deve considerar as necessidades do negócio, pois diferentes contextos podem valorizar mais a interpretabilidade do que desempenho puro em alguma métrica técnica.

In [21]:
# define pesos para cada criterio de selecao do modelo campeao
selection_policy = {
    "technical_score": {
        "roc_auc_teste": 0.35,
        "recall_inadimplente": 0.25,
        "precision_inadimplente": 0.15,
        "f1_inadimplente": 0.15,
        "stability_score": 0.10, # vai depender da validacao cruzada (cv), desvio padrao do roc_auc
    },
    "final_score": {
        "technical_score": 0.75,
        "interpretability_score": 0.15,
        "operational_simplicity_score": 0.10, # custo computacional, facilidade de deploy, facilidade de monitoramento, etc.
    },
    "notes": (
        "Pesos adotados para seleção do modelo campeão nesse projeto. "
        "Em um projeto real, esses pesos devem ser definidos "
        "considerando negócio, risco, compliance e engenharia."
    ),
}

selection_policy

{'technical_score': {'roc_auc_teste': 0.35,
  'recall_inadimplente': 0.25,
  'precision_inadimplente': 0.15,
  'f1_inadimplente': 0.15,
  'stability_score': 0.1},
 'final_score': {'technical_score': 0.75,
  'interpretability_score': 0.15,
  'operational_simplicity_score': 0.1},
 'notes': 'Pesos adotados para seleção do modelo campeão nesse projeto. Em um projeto real, esses pesos devem ser definidos considerando negócio, risco, compliance e engenharia.'}

In [22]:
# atribui scores de interpretabilidade e simplicidade operacional para cada modelo
interpretability_score                      = {"logistic_regression": 1.00, "decision_tree": 0.80, "random_forest": 0.50}
operational_simplicity_score                = {"logistic_regression": 1.00, "decision_tree": 0.90, "random_forest": 0.65}

df_decision["interpretability_score"]       = (df_decision["model_name"].map(interpretability_score).fillna(0.50))
df_decision["operational_simplicity_score"] = (df_decision["model_name"].map(operational_simplicity_score).fillna(0.50))
df_decision["stability_score"]              = (1 - df_decision["roc_auc_cv_desvio"].clip(lower=0, upper=1)) # quanto menor o desvio padrao do roc_auc na validacao cruzada, maior a estabilidade do modelo

df_decision

,run_id,run_name,model_name,roc_auc_teste,roc_auc_cv_media,roc_auc_cv_desvio,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente,interpretability_score,operational_simplicity_score,stability_score
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369,1.0,1.00,0.998905
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779,0.5,0.65,0.998372
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638,0.8,0.90,0.995549


### 2.3 Calculando _scores_ e selecionando o modelo candidato a campeão

E finalmente calculamos os _scores_ e determinamos o modelo campeão:

In [23]:
# calcula score tecnico e score final para cada modelo candidato, considerando pesos definidos na politica de selecao
df_decision["technical_score"] = df_decision.apply(lambda row: sum([v * row[k] for k, v in selection_policy["technical_score"].items()]), axis=1)
df_decision["final_score"]     = df_decision.apply(lambda row: sum([v * row[k] for k, v in selection_policy["final_score"].items()]), axis=1)

# gera ranking final dos modelos candidatos, considerando score final e desempenho no conjunto de teste
df_decision_ranking = (
    df_decision
    .sort_values(["final_score", "roc_auc_teste"], ascending=False)
    .reset_index(drop=True)
)

df_decision_ranking

,run_id,run_name,model_name,roc_auc_teste,roc_auc_cv_media,roc_auc_cv_desvio,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente,interpretability_score,operational_simplicity_score,stability_score,technical_score,final_score
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369,1.0,1.00,0.998905,0.759554,0.819665
1,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638,0.8,0.90,0.995549,0.757738,0.778303
2,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779,0.5,0.65,0.998372,0.756847,0.707635


In [24]:
# define o modelo campeao como o primeiro colocado no ranking final, considerando score final e desempenho no conjunto de teste
df_champion = df_decision_ranking.iloc[0]
champion    = df_champion.to_dict()

df_champion[['model_name', 'run_name', 'run_id']]

model_name                          logistic_regression
run_name      hibrid_log_comparison_logistic_regression
run_id                 e4afeca8be834b3fb69b1dbd6e3f4c0f
Name: 0, dtype: object

### 2.4 Documentando o modelo campeão

Escrevemos o resumo reunindo informações necessárias para justificar a escolha do modelo candidato a campeão, temos: o contexto, o modelo selecionado, as métricas principais, o critério de seleção (pesos técnicos e o _score_ final) e a justificativa.

Outras informações que também costumam ser incluídas no resumo da decisão são: as limitações, os riscos, quais dados foram utilizados, métricas por segmento, análise de falhas e critérios de monitoramento.

In [9]:
champion_summary = (
    f"# Seleção do modelo campeão\n"
    f"\n"
    f"## Contexto\n"
    f"\n"
    f"Projeto: risco de crédito \n"
    f"Tarefa: classificação binária \n"
    f"Target: inadimplente \n"
    f"Dataset version: synthetic_credit_v1 \n"
    f"Feature set: core_features_v1 \n"
    f"\n"
    f"## Modelo selecionado\n"
    f"\n"
    f"- Modelo: {champion['model_name']}\n"
    f"- Run name: {champion['run_name']}\n"
    f"- Run ID: {champion['run_id']}\n"
    f"\n"
    f"## Principais métricas\n"
    f"\n"
    f"- ROC-AUC teste: {champion["roc_auc_teste"]:.4f}\n"
    f"- ROC-AUC CV média: {champion["roc_auc_cv_media"]:.4f}\n"
    f"- ROC-AUC CV desvio: {champion["roc_auc_cv_desvio"]:.4f}\n"
    f"- Precision inadimplente: {champion["precision_inadimplente"]:.4f}\n"
    f"- Recall inadimplente: {champion["recall_inadimplente"]:.4f}\n"
    f"- F1 inadimplente: {champion["f1_inadimplente"]:.4f}\n"
    f"\n"
    f"## Critérios de decisão\n"
    f"\n"
    f"A seleção considerou desempenho técnico, estabilidade, interpretabilidade e simplicidade operacional.\n"
    f"\n"
    f"Pesos técnicos:\n"
    f"- ROC-AUC teste: 35%\n"
    f"- Recall inadimplente: 25%\n"
    f"- Precision inadimplente: 15%\n"
    f"- F1 inadimplente: 15%\n"
    f"- Estabilidade: 10%\n"
    f"\n"
    f"Score final:\n"
    f"- Score técnico: 75%\n"
    f"- Interpretabilidade: 15%\n"
    f"- Simplicidade operacional: 10%\n"
    f"\n"
    f"## Justificativa\n"
    f"\n"
    f"O modelo selecionado apresentou o melhor equilíbrio entre desempenho e critérios operacionais dentro da política de seleção definida.\n"
    f"\n"
    f"## Próximo passo\n"
    f"\n"
    f"Preparar este modelo para registro no MLflow Model Registry.\n"
)

print(champion_summary)

# Seleção do modelo campeão

## Contexto

Projeto: risco de crédito 
Tarefa: classificação binária 
Target: inadimplente 
Dataset version: synthetic_credit_v1 
Feature set: core_features_v1 

## Modelo selecionado

- Modelo: logistic_regression
- Run name: hibrid_log_comparison_logistic_regression
- Run ID: e4afeca8be834b3fb69b1dbd6e3f4c0f

## Principais métricas

- ROC-AUC teste: 0.9036
- ROC-AUC CV média: 0.9023
- ROC-AUC CV desvio: 0.0011
- Precision inadimplente: 0.3890
- Recall inadimplente: 0.8232
- F1 inadimplente: 0.5284

## Critérios de decisão

A seleção considerou desempenho técnico, estabilidade, interpretabilidade e simplicidade operacional.

Pesos técnicos:
- ROC-AUC teste: 35%
- Recall inadimplente: 25%
- Precision inadimplente: 15%
- F1 inadimplente: 15%
- Estabilidade: 10%

Score final:
- Score técnico: 75%
- Interpretabilidade: 15%
- Simplicidade operacional: 10%

## Justificativa

O modelo selecionado apresentou o melhor equilíbrio entre desempenho e critérios operac

E finalmente podemos registrar a decisão diretamente na _run_ vencedora, adicionando _tags_ e artefatos. Instânciando um _client_ com [`MlflowClient`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html) para sermos capazes de adicionar novas _tags_ nessa run específica e também salvar artefatos com a decisão, a política de seleção e a matriz final:

In [ ]:
MODEL_NAME = "credit_risk_classifier"
client     = mlflow.MlflowClient()

client.set_tag(champion['run_id'], "selection_status",      "champion_candidate")                   # indica que o modelo foi selecionado como candidato a campeao
client.set_tag(champion['run_id'], "registry_model_name",   MODEL_NAME)                             # indica nome do modelo
client.set_tag(champion['run_id'], "selection_rule",        "weighted_score_v1")                    # indica regra de selecao usada para escolher o modelo campeao
client.set_tag(champion['run_id'], "selected_for_registry", "true")                                 # indica que o modelo foi selecionado para registro

mlflow.log_text(champion_summary,     "selection/champion_summary.md", run_id=champion['run_id'])   # explica porque o modelo foi selecionado como campeao
mlflow.log_dict(selection_policy,     "selection/selection_policy.json", run_id=champion['run_id']) # registra politica que levou a escolha do modelo campeao
mlflow.log_table(df_decision_ranking, "selection/decision_matrix.json", run_id=champion['run_id'])  # registra matriz de decisao dos modelos candidatos

Além dos artefatos acima, adicionamos também um artefato para a "preparação do registro", que é um exercício de organização e governança. O **Model Registry** é o local onde os modelos são promovidos para serem usados em produção. Não devemos simplesmente "jogar" um modelo lá, é preciso garantir que seja possível determinar exatamente de onde o modelo veio, por que ele foi escolhido e quais critérios foram usados para validá-lo.

In [ ]:
MODEL_ARTIFACT_PATH  = "model"

# path dentro da run onde o artefato do modelo esta salvo, local que o MLflow usara para move-lo para o registro
champion_model_uri   = f"runs:/{champion['run_id']}/{MODEL_ARTIFACT_PATH}"

# prepara metadados para registro do modelo campeao no MLflow Model Registry
registry_preparation = {
    "registered_model_name":      MODEL_NAME,
    "source_run_id":              champion['run_id'],
    "source_run_name":            champion['run_name'],
    "model_name":                 champion['model_name'],
    "model_uri":                  champion_model_uri,
    "intended_stage_next_lesson": "Staging",
    "selection_status":           "champion_candidate",
    "notes":                      "Metadados preparados registro efetivo no Model Registry."
}

# registra os metadados na run do modelo campeao no MLflow
mlflow.log_dict(registry_preparation, "selection/registry_preparation.json", run_id=champion['run_id'])

registry_preparation

{'registered_model_name': 'credit_risk_classifier',
 'source_run_id': 'e4afeca8be834b3fb69b1dbd6e3f4c0f',
 'source_run_name': 'hibrid_log_comparison_logistic_regression',
 'model_name': 'logistic_regression',
 'model_uri': 'runs:/e4afeca8be834b3fb69b1dbd6e3f4c0f/model',
 'intended_stage_next_lesson': 'Staging',
 'selection_status': 'champion_candidate',
 'notes': 'Metadados preparados registro efetivo no Model Registry.'}

Nessa preparação, definimos o nome para o modelo registrado (`"credit_risk_classifier"`) e a variável `champion_model_uri` que aponta para o artefato do modelo dentro da _run_ vencedora (a URI assume que ele foi salvo na pasta `model` pelo autolog).

## 3 Model Registry

Até agora estavamos trabalhando com o conceito de _run_ (execução), que é o registro histórico de experimentos. Com as _runs_ somos capazes de documentar o que aconteceu durante um experimento específico, registrando seus parâmetros, métricas, artefatos e o código utilizado. Durante o desenvolvimento de um modelo, é comum ter várias _runs_, mas nem todas se tornarão modelos úteis para o negócio.

Agora introduzimos o **Model Registry**, que registra quais modelos foram validados e aprovados para uso. Ele atua como um gerenciador de versões, onde o foca deixa de ser o que aconteceu nos treinos e passa a ser o gerenciamento da evolução do modelo ao longo do tempo.

Alguns conceitos importantes:

- O Model Registry permite que os modelos sejam versionados (v1, v2, v3), independentemente de quantas _runs_ foram executadas. Sempre que um modelo for registrado com o mesmo nome, seu versionamento é atualizado
- É possível utilizar _aliases_ (mutáveis), como `production` ou `staging`, para apontar para uma versão específica, facilitando a atualização de modelos sem precisar alterar o código da aplicação. O registro permite adicionar descrições, _tags_ e metadados que facilitam a governança, algo que vai além dos dados de um único treino armazenado em uma run.

Resumindo, a _run_ é a origem, o rastro do que foi treinado, e o Model Registry é o destino, o catálogo onde são consolidados os modelos que serão efetivamente usados em produção.

### 3.1 Levantando servidor local do MLflow

Para simular um ambiente real, vamos levantar o servidor do MLflow, dessa forma o código passará a interagir com um "serviço externo". Além disso, o servidor vai atuar como um coordenador que gerencia o acesso ao banco de dados, garantindo a integridade, o versionamento e o controle de estados (_aliases_ e _tags_) dos modelos, algo que o salvamento local simples não provê. Então rodamos o comando abaixo no terminal:

```terminal
mlflow server --backend-store-uri sqlite:///./mlflow.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000
```

Esse comando informa que deve ser usado o banco SQLite `mlflow.db` como fonte de consulta e para armazenamento dos metadados e a pasta `mlartifacts` para armazenar os artefatos dos modelos registrados. Note que ambos os arquivos são referenciados na raiz do projeto, portanto devemos verificar se estamos rodando o comando na raiz do projeto.

Após isso, precisamos fazer a ligação do nosso código com o servidor:

In [31]:
mlflow.set_tracking_uri("http://127.0.0.1:5000") # agora aponta URI do backend para o servidor MLflow
mlflow.get_experiment_by_name(EXPERIMENT_NAME)   # define como ativo experimento da comparacao dos modelos

<Experiment: artifact_location='/home/bruno/estudos/data-science-portifolio/mlflow/mlruns/1/', creation_time=1791835030069, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1791835030069, lifecycle_stage='active', name='risco_credito_comparacao_modelos', tags={}, trace_location=None, workspace='default'>

Com isso, os códigos rodando no notebook não precisam mais saber da existência do banco de dados SQLite. Agora o armazenamento dos metadados serão feitos por a partir de requisições HTTP feitas para o servidor, que por sua vez, é quem decide como gravar esses dados.

### 3.2 Dependências e configuração

In [1]:
# importa dependencias
import sys
sys.path.insert(0, "../src")

from support_module import RANDOM_STATE, generate_credit_risk_base, data_preprocess, profile_for_test_prediction
import pandas as pd
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score#, classification_report, confusion_matrix, RocCurveDisplay

In [13]:
# gera base de dados sintetica para risco de credito
df = generate_credit_risk_base(6000)
print(df.shape)
df.head()

(6000, 9)


,idade,renda_mensal,tempo_emprego_anos,score_interno,valor_solicitado,num_parcelas,tipo_renda,possui_restricao,inadimplente
0,25,5630.15,0.3,501,5234.99,24,CLT,nao,0
1,56,2491.57,9.6,593,8264.73,18,CLT,sim,0
2,50,1020.17,1.0,695,12435.70,12,CLT,nao,0
3,41,1580.32,11.7,527,14883.12,48,CLT,nao,0
4,41,6635.91,2.7,618,12856.88,24,CLT,nao,0


In [14]:
# seleciona features
TARGET   = "inadimplente"
FEATURES = [c for c in df.columns if c != TARGET]
x        = df[FEATURES]
y        = df[TARGET]

# separa colunas numericas e categoricas
num_cols = df[FEATURES].select_dtypes(include='number').columns
cat_cols = [f for f in FEATURES if f not in num_cols]

# separa dados de treino e teste, seguindo proporcao de casos positivos e negativos do target
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# pipeline de transformacao dos daddos (disponivel em `src/support_module.py`)
preprocess = data_preprocess()

x_train.shape, x_test.shape

((4500, 8), (1500, 8))

### 3.3 Reproduzindo o modelo campeão do zero

Já selecionamos o modelo campeão na seção 2, mas, vamos treiná-lo novamente agora usando o _tracking_ com o servidor local. Assim, teremos uma _run_, com parâmetros, métricas, _tags_ e artefatos, ligada a uma versão registrada do modelo. Isso também evita reutilizar um objeto já treinado e garante que a versão registrada será criada a partir de uma execução limpa.

Criamos uma função auxiliar que retorna um clone do estimador:

In [18]:
# funcao auxiliar, evita reutilizar objeto ja treinado, garantindo que o modelo seja instanciado do zero
def build_estimator(model_name):
    model_candidates = {
        "logistic_regression": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "decision_tree": DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            min_samples_leaf=20,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    }

    if model_name not in model_candidates:
        raise ValueError(f"Modelo não reconhecido: {model_name}")

    return clone(model_candidates[model_name])

In [29]:
# modelo campeao selecionado na etapa anterior, que sera registrado no MLflow Model Registry
CHAMPION_MODEL_NAME = champion['model_name']
CHAMPION_MODEL_NAME

'logistic_regression'

In [ ]:
# nome do modelo definido na etapa anterior,que sera registrado no MLflow Model Registry
MODEL_NAME

'credit_risk_classifier'

### 3.4 Registrando o modelo campeão no Model Registry

Vamos registrar o modelo através de uma _run_ muito similar a outras, apenas com algumas diferenças:

- **Contrato de dados:** ao passarmos uma amostra dos dados de treino (_input_), e sua predição (_output_ do modelo), para o método `infer_signature`, ele é capaz de inferir os tipos de dados e nomes das colunas esperados para a entrada e saída do modelo. Com isso, quando o modelo for implantado, o MLflow usará essa assinatura para validar se os dados que estão chegando na API estão no formato correto. Se forem enviados dados de tipo diferente, ou com uma coluna faltando, o sistema poderá identificar o erro antes mesmo de tentar rodar a inferência.
- **Modelo completo armazenado como artefato:** a função `mlflow.sklearn.log_model` persiste o modelo no local definido para os artefatos registrados, e não como um simples arquivo binário, mas sim como um "pacote completo".
  - Configurações de ambiente: um arquivo `conda.yaml` ou `requirements.txt`, para que, quando o modelo for carregado em outro ambiente, o MLflow saiba quais bibliotecas e versões são necessárias para que ele funcione
  - Objeto serializado: o pipeline treinado é transformado em um formato binário (geralmente usando pickle ou similar) que pode ser lido posteriormente
  - Metadados adicionais: o contrato de dados é anexado ao pacote do modelo.

In [ ]:
# define experimento para registro do modelo campeao no MLflow Model Registry
mlflow.set_experiment("model_registry")

# instancia clone do estimador
champion_estimator = build_estimator(CHAMPION_MODEL_NAME)

# roda pipeline de preparacao dos dados
champion_pipe = Pipeline(
    steps=[
        ("prep", data_preprocess()),
        ("model", champion_estimator),
    ]
)

input_example = x_train.head(5).copy()

with mlflow.start_run(run_name="register_champion") as run:
    champion_pipe.fit(x_train, y_train)

    y_pred  = champion_pipe.predict(x_test)
    y_proba = champion_pipe.predict_proba(x_test)[:,1]

    # registra metricas da run
    mlflow.log_metrics({
        "roc_auc_teste":          float(roc_auc_score(y_test, y_proba)),
        "accuracy_teste":         float(accuracy_score(y_test, y_pred)),
        "precision_inadimplente": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall_inadimplente":    float(recall_score(y_test, y_pred, zero_division=0)),
        "f1_inadimplente":        float(f1_score(y_test, y_pred, zero_division=0))
    })

    # registra parametros da run
    mlflow.log_params({
        "model_name":            CHAMPION_MODEL_NAME,
        "registered_model_name": MODEL_NAME,
        "dataset_version":       "synthetic_credit_v1",
        "feature_set":           "core_features_v1",
        "target":                TARGET
    })

    # registra tags da run
    mlflow.set_tags({
        "project":          "risco_credito",
        "run_purpose":      "register_champion_model",
        "selection_status": "champion_candidate",
        "tracking_backend": "sqlite_local"
    })

    # define um contrato de dados para input e output do modelo
    signature = infer_signature(
        input_example,
        champion_pipe.predict(input_example)
    )

    # salva modelo com parametros necessarios para funcionar bem
    model_info = mlflow.sklearn.log_model(
        sk_model=champion_pipe,            # modelo treinado, pronto para ser salvo como artefato da run
        name="model",                      # define indentificador interno do artefato do modelo dentro da run
        signature=signature,               # vincula contrato de dados ao modelo, garantindo que o input e output do modelo sejam validados
        input_example=input_example,       # salva amostra de input para o modelo
        registered_model_name=MODEL_NAME,  # cria ou atualiza modelo no MLflow Model Registry com o nome definido em `MODEL_NAME`
        skops_trusted_types=True,          # "The saved sklearn model references untrusted types. (...) set the 'skops_trusted_types' parameter when calling 'log_model' (...): ['numpy.dtype'].
        serialization_format='cloudpickle' # "The saved sklearn model could not be serialized (...) set the serialization_format 'cloudpickle' (...).
    )

    run_id = run.info.run_id

print("Run ID:", run_id)
print("Model URI:", model_info.model_uri)

/home/bruno/estudos/data-science-portifolio/mlflow/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/14 22:38:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechani

🏃 View run register_champion at: http://127.0.0.1:5000/#/experiments/4/runs/fa37e584163b4a8a96d34a3073f93ada
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Run ID: fa37e584163b4a8a96d34a3073f93ada
Model URI: models:/m-68250cad764d4b0d8b535a7c0718d9ec


Agora que temos um modelo registrado no Model Registry, podemos consultar a versão do modelo via `client` do MLflow:

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

model_versions = client.search_model_versions(
    f"name = '{MODEL_NAME}'" # busca pelo nome do modelo registrado no MLflow Model Registry
)

registered_version = max(
    [mv for mv in model_versions if mv.run_id == run_id],
    key=lambda mv: int(mv.version)
)

print("Registered model:", registered_version.name)
print("Version:",       registered_version.version)
print("Run ID:",         registered_version.run_id)
print("Source:",         registered_version.source)
print("Status:",         registered_version.status)

Registered model: credit_risk_classifier
Version: 1
Run ID: fa37e584163b4a8a96d34a3073f93ada
Source: models:/m-68250cad764d4b0d8b535a7c0718d9ec
Status: READY


In [37]:
model_versions

[<ModelVersion: aliases=[], creation_timestamp=1792028369703, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1792028369703, metrics=None, model_id=None, name='credit_risk_classifier', params=None, run_id='fa37e584163b4a8a96d34a3073f93ada', run_link='', source='models:/m-68250cad764d4b0d8b535a7c0718d9ec', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>]

### 3.5 Atualziando as descrições do modelo e da versão

Agora podemos adicionar a descrição do modelo e acrescentar contexto em relação à versão: atualizar os metadados do registro do modelo, da versão do modelo e as _tags_, tanto do projeto quanto da _task_ em si, que é o classificador para risco de crédito. A _task_ é uma classificação binária para prever se um cliente será inadimplente ou adimplente. 

A versão do conjunto de dados, que neste caso é uma base sintética, continua na primeira versão, pois é a única utilizada até agora. 

Por fim, é definido o ciclo de vida da versão como "candidata". O ciclo de vida começa como `candidate` porque o modelo ainda não foi promovido. A documentação do `MLflowClient` inclui métodos para atualizar descrições, definir _tags_ no modelo registrado e definir _tags_ em versões específicas do modelo.

In [ ]:
# Adicionando descrição e tags ao Registry
model_description = (
    "Modelo de classificação de risco de crédito usado no projeto didático do curso.\n"
    "\n"
    "Este registered model agrupa versões candidatas do classificador de inadimplência.\n"
    "As versões devem ser promovidas por aliases e tags de ciclo de vida."
)

version_description = (
    f"Versão registrada a partir da run {run_id}.\n"
    f"\n"
    f"Modelo selecionado na etapa de model selection do projeto.\n"
    f"Algoritmo: {CHAMPION_MODEL_NAME}\n"
    f"Dataset: synthetic_credit_v1\n"
    f"Feature set: core_features_v1"
)

client.update_registered_model(name=MODEL_NAME, description=model_description)
client.update_model_version(   name=MODEL_NAME, description=version_description, version=registered_version.version)

client.set_registered_model_tag(MODEL_NAME, "project", "risco_credito")
client.set_registered_model_tag(MODEL_NAME, "task", "binary_classification")

client.set_model_version_tag(MODEL_NAME, registered_version.version, "dataset_version",  "synthetic_credit_v1")
client.set_model_version_tag(MODEL_NAME, registered_version.version, "model_name",       CHAMPION_MODEL_NAME)
client.set_model_version_tag(MODEL_NAME, registered_version.version, "lifecycle_status", "candidate") # modelo ainda eh candidato a campeao, precisa ser promovido para Staging ou Production

### 3.6 Definindo _aliases_

Agora vamos criar dois _aliases_ diferentes, que neste momento, vão apontar para a mesma versão do modelo: `candidate` e `champion_candidate`. Essa é uma prática comum e útil no gerenciamento de modelos por alguns motivos:

- Camadas de Semântica:
  - O _alias_ `candidate` pode ser uma convenção para indicar que o modelo está no "repositório de candidatos", aguardando revisão.
  - O _alias_ `champion_candidate` carrega uma informação semântica extra: este modelo não é apenas um candidato qualquer, mas aquele específico que venceu a etapa de seleção.
- Flexibilidade de Referência: Uma "versão 2" do modelo pode ser criada e promovida para o _alias_ `candidate` enquanto o _alias_ `champion_candidate` continua apontando para a "versão 1" (o campeão original) para fins de comparação ou histórico.
- Desacoplamento do Código: A aplicação de inferência pode ser configurada para buscar o modelo pelo _alias_ `champion_candidate`, e assim ela sempre pegará a última versão promovida para esse _alias_.

_Aliases_ são ponteiros mutáveis, e diferente do número da versão (que é imutável), podem ser movidos. Hoje, ambos apontam para a versão 1, mas amanhã o _alias_ `candidate` pode passar a apontar para outra versão, deixando o _alias_ `champion_candidate` onde está ou movendo-o também.

Ao usar múltiplos _aliases_, são criados diferentes "pontos de entrada" para o modelo, permitindo que diferentes processos apontem para versões diferentes conforme a necessidade, sem precisar alterar uma única linha de código na aplicação final.

In [39]:
# Adicionando aliases iniciais
client.set_registered_model_alias(name=MODEL_NAME, version=registered_version.version, alias="candidate")
client.set_registered_model_alias(name=MODEL_NAME, version=registered_version.version, alias="champion_candidate")

alias_candidate_version          = client.get_model_version_by_alias(MODEL_NAME, "candidate")
alias_champion_candidate_version = client.get_model_version_by_alias(MODEL_NAME, "champion_candidate")

print("Alias `candidate` aponta para a versão:", alias_candidate_version.version)
print("Alias `champion_candidate` aponta para a versão:", alias_champion_candidate_version.version)

Alias `candidate` aponta para a versão: 1
Alias `champion_candidate` aponta para a versão: 1


## 4 Ciclo de vida do modelo

Agora que temos um modelo registrado, podemos seguir para o processo de promoção do modelo.

O **ciclo de vida** de um modelo compreende as etapas: candidato (_champion\_candidate_) -> homologação (_staging_) -> produção (_production_) -> arquivado (_archived_). O ideal é que esse fluxo seja gerenciado com o uso de _aliases_, ao invés de referenciar os modelos pelo número de sua versão, assim temos referências estáveis associando um _alias_ como _"staging"_ ou _"production"_ a uma versão específica. Isso permite que o código de aplicação (que carrega o modelo) permaneça o mesmo, mesmo que o modelo seja atualizado.

Também pomodes associar uso de _tags_ para a governança, descrevendo o estado real do modelo, como a indicação da etapa em que o modelo se encontra, registrando o progresso de validações (como 'pending_business_validation' ou 'approved_for_production') ou definindo políticas sobre o que deve acontecer com versões antigas ou superadas.

In [ ]:
# candidato -> staging (homologação) -> production

In [ ]:
# MODEL_NAME = 'credit_risk_classifier'
# run_id = 'fa37e584163b4a8a96d34a3073f93ada'

In [ ]:
# import mlflow

# # define a ligacao com o banco de dados relacional ja existente, onde o MLflow vai armazenar os metadados do experimento
# mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# # recupera experimento pelo nome
# EXPERIMENT_NAME = "risco_credito_comparacao_modelos"
# experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)


In [ ]:
# from mlflow import MlflowClient

# client = MlflowClient()

# model_versions = client.search_model_versions(
#     f"name = '{MODEL_NAME}'" # busca pelo nome do modelo registrado no MLflow Model Registry
# )

# registered_version = max(
#     [mv for mv in model_versions if mv.run_id == run_id],
#     key=lambda mv: int(mv.version)
# )

# print("Registered model:", registered_version.name)
# print("Version:",       registered_version.version)
# print("Run ID:",         registered_version.run_id)
# print("Source:",         registered_version.source)
# print("Status:",         registered_version.status)

Registered model: credit_risk_classifier
Version: 1
Run ID: fa37e584163b4a8a96d34a3073f93ada
Source: models:/m-68250cad764d4b0d8b535a7c0718d9ec
Status: READY


### 4.1 Promovendo modelo para homologação (_staging_)

Nessa etapa apontamos o _alias_ `staging` para a versão registrada do modelo candidato a campeão e adicionamos _tags_ para descrever seu estado.

In [ ]:
# promovendo modelo para staging
MODEL_VERSION = registered_version.version
print("Versão do modelo campeao (que está registrada):", MODEL_VERSION)

# aponta alias "staging" para versao campea, promovendo modelo para homologacao
client.set_registered_model_alias(name=MODEL_NAME, alias="staging", version=MODEL_VERSION)

# aplica tags para indicar etapa e status de validacao
client.set_model_version_tag(MODEL_NAME, MODEL_VERSION, "lifecycle_status", "staging")
client.set_model_version_tag(MODEL_NAME, MODEL_VERSION, "validation_status", "pending_business_validation")

# recupera modelo pelo alias "staging" para confirmar que a promocao foi realizada corretamente
staging_version = client.get_model_version_by_alias(MODEL_NAME, "staging")

print("Alias 'staging' aponta para a versão:", staging_version.version)

Versão do modelo campeao (que está registrada): 1
Alias 'staging' aponta para a versão: 1


### 4.2 Validando modelo em _staging_

Antes de colocar o modelo em produção, é essencial carregar a versão apontada pelo _alias_ e realizar uma inferência local (usando o modelo carregado via URI `models:/MODEL_NAME@staging`). Dessa forma podemos validar o modelo e garantir que a versão esperada está funcionando corretamente.

Note que como estamos apontando para o _alias_ `staging`, conforme os modelo for sendo atualizado, basta atualizar o alias que o código abaixo não precisa ser alterado.

In [ ]:
novo_cliente       = profile_for_test_prediction()
staging_model_uri  = f"models:/{MODEL_NAME}@staging"
staging_model      = mlflow.pyfunc.load_model(staging_model_uri) # usa interface generica para carregar o modelo
staging_prediction = staging_model.predict(novo_cliente)

print("Modelo carregado de:", staging_model_uri)
print("Predição:", staging_prediction)

Modelo carregado de: models:/credit_risk_classifier@staging
Predição: [0]


Acima foi utilizado o `mlflow.pyfunc.load_model` para carregar o modelo, e com isso temos acesso ao método `predict()`, que executa o método de predição do modelo conforme ele foi treinado (e registrado), retornando o resultado da predição. Ou seja, nesse caso, o _threshold_ está "embutido" dentro do artefato do modelo e é utilizado para classificar os resultados em `0` ou `1`.

Mas como o modelo foi registrado com o _flavor_ sklearn, também é possível carregá-lo com `mlflow.sklearn.load_model()`, que nos dá acesso a métodos como o `predict_proba()` que retorna as probabilidades de cada resultado.

In [ ]:
sklearn_model = mlflow.sklearn.load_model(staging_model_uri)
probabilidade = sklearn_model.predict_proba(novo_cliente)[0, 1]

print(f"Probabilidade estimada de inadimplência: {probabilidade:.1%}")

Probabilidade estimada de inadimplência: 27.2%


### 4.3 Promovendo modelo para produção

Nessa etapa apontamos o _alias_ `production` para a versão atualmente em `staging` do modelo campeão e adicionamos _tags_ para descrever seu estado.

In [ ]:
# promovendo para production
MODEL_VERSION = registered_version.version
print("Versão do modelo em 'staging':", MODEL_VERSION)

# aponta aliases "production" e "champion" para versao campea, promovendo modelo para producao
client.set_registered_model_alias(name=MODEL_NAME, alias="production", version=MODEL_VERSION)
client.set_registered_model_alias(name=MODEL_NAME, alias="champion", version=MODEL_VERSION)

# aplica tags para indicar etapa e status de validacao
client.set_model_version_tag(MODEL_NAME, MODEL_VERSION, "lifecycle_status", "production")
client.set_model_version_tag(MODEL_NAME, MODEL_VERSION, "validation_status", "approved_for_production")
client.set_model_version_tag(MODEL_NAME, MODEL_VERSION, "archive_policy", "keep_until_replaced_by_new_champion")

# recupera modelo pelo alias "production" para confirmar que a promocao foi realizada corretamente
production_version = client.get_model_version_by_alias(MODEL_NAME, "production")

print("Alias 'production' aponta para a versão:", production_version.version)

Versão do modelo em 'staging': 1
Alias 'production' aponta para a versão: 1


In [12]:
# validando a versao de producao antes de servir
print("Modelo:", production_version.name)
print("Versão de produção:", production_version.version)
print("Run de origem:", production_version.run_id)
print("Source:", production_version.source)

production_model_uri  = f"models:/{MODEL_NAME}@production"
production_model      = mlflow.pyfunc.load_model(production_model_uri)
production_prediction = production_model.predict(novo_cliente)

print("Modelo carregado de:", production_model_uri)
print("Predição:", production_prediction)

Modelo: credit_risk_classifier
Versão de produção: 1
Run de origem: fa37e584163b4a8a96d34a3073f93ada
Source: models:/m-68250cad764d4b0d8b535a7c0718d9ec
Modelo carregado de: models:/credit_risk_classifier@production
Predição: [0]
